# Module Transform — Nettoyage et structuration des données

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haja171106/donnee2-aqi/blob/feat/notebooks-analysis/notebooks/transform.ipynb)

Ce notebook analyse et documente le module `src/transform.py`.

**Rôle :** Lire tous les fichiers JSON bruts de `data/raw/`, les transformer en un unique fichier CSV propre dans `data/clean/qualite_air.csv`.

**Principes :**
- **Idempotence** : le résultat ne dépend que de `raw/`, pas du précédent `clean/`
- **Reconstruction totale** : le CSV est entièrement réécrit à chaque run
- **Déduplication** : une seule ligne par couple (ville, timestamp)

## 1. Dépendances et configuration

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd

from config import RAW_DIR, CLEAN_FILE

print("Toutes les dépendances sont chargées ✓")

## 2. Analyse des fonctions

### `_rows_from_file(path) -> list[dict]`

Ouvre un fichier JSON brut et extrait chaque mesure horaire sous forme de dictionnaire.

**Étapes :**
1. Charge le fichier JSON
2. Récupère les métadonnées de la ville depuis `_city_meta`
3. Pour chaque élément de la clé `list` (une mesure par heure) :
   - Convertit le timestamp UNIX en ISO 8601
   - Extrait l'AQI (`main.aqi`)
   - Extrait les 8 polluants (`components.*`)
4. Retourne une liste de dictionnaires

**Structure produite :**
```
{'ville': 'Paris', 'pays': 'FR', 'latitude': 48.8566, 'longitude': 2.3522,
 'timestamp_utc': '2026-07-24T14:25:44+00:00', 'aqi': 2,
 'co': 250.0, 'no': 0.5, 'no2': 15.0, 'o3': 60.0, 'so2': 2.0,
 'pm2_5': 8.0, 'pm10': 12.0, 'nh3': 1.2}
```

In [ ]:
COMPONENT_COLS = ["co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]

def _rows_from_file(path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    city_meta = payload.get("_city_meta", {})
    rows = []
    for item in payload.get("list", []):
        dt = datetime.fromtimestamp(item["dt"], tz=timezone.utc)
        row = {
            "ville": city_meta.get("name"),
            "pays": city_meta.get("country"),
            "latitude": city_meta.get("lat"),
            "longitude": city_meta.get("lon"),
            "timestamp_utc": dt.isoformat(),
            "aqi": item.get("main", {}).get("aqi"),
        }
        components = item.get("components", {})
        for col in COMPONENT_COLS:
            row[col] = components.get(col)
        rows.append(row)
    return rows

print("Fonction _rows_from_file définie ✓")

### `rebuild_clean() -> pd.DataFrame`

Fonction principale qui :
1. Parcourt **tous** les fichiers `*.json` de `data/raw/`
2. Extrait les lignes via `_rows_from_file`
3. Les assemble dans un DataFrame pandas
4. **Déduplique** sur (ville, timestamp_utc)
5. **Trie** chronologiquement par ville
6. **Écrit** `data/clean/qualite_air.csv`

**Pourquoi une reconstruction totale ?**
Plutôt que d'ajouter les nouvelles lignes à la fin (append), on reconstruit tout depuis zéro. Cela garantit qu'aucune anomalie ne s'accumule entre les runs et simplifie la déduplication.

In [ ]:
def rebuild_clean() -> pd.DataFrame:
    all_rows = []
    for path in sorted(RAW_DIR.glob("*.json")):
        all_rows.extend(_rows_from_file(path))

    df = pd.DataFrame(all_rows)
    if df.empty:
        print("[transform] Aucune donnée brute trouvée — clean/ non modifié.")
        return df

    df = df.drop_duplicates(subset=["ville", "timestamp_utc"])
    df = df.sort_values(["ville", "timestamp_utc"]).reset_index(drop=True)

    df.to_csv(CLEAN_FILE, index=False)
    print(f"[transform] {len(df)} lignes écrites dans {CLEAN_FILE}")
    return df

print("Fonction rebuild_clean définie ✓")

## 3. Analyse du fichier clean produit

Regardons la structure et les statistiques du fichier `qualite_air.csv`.

In [ ]:
df = pd.read_csv(CLEAN_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes\n")
print("Colonnes :")
for col in df.columns:
    print(f"  • {col}")
print()
print(f"Types :\n{df.dtypes}\n")

In [ ]:
print("Aperçu des 5 premières lignes :")
df.head(5)

In [ ]:
print("Statistiques descriptives :")
df.describe()

In [ ]:
print("Nombre de lignes par ville :")
df["ville"].value_counts()

In [ ]:
# Vérification de la déduplication
duplicates = df.duplicated(subset=["ville", "timestamp_utc"]).sum()
print(f"Doublons sur (ville, timestamp_utc) : {duplicates}")

# Vérification du tri chronologique par ville
tri_ok = True
for ville, grp in df.groupby("ville"):
    if not grp["timestamp_utc"].is_monotonic_increasing:
        print(f"  ⚠ {ville} : timestamps non triés")
        tri_ok = False

if tri_ok:
    print("✓ Toutes les villes ont leurs timestamps triés chronologiquement")

# Période couverte
print(f"\nPériode : {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"Villes uniques : {df['ville'].nunique()}")
print(f"Polluants présents : {[c for c in COMPONENT_COLS if c in df.columns]}")

## 4. Exemple : transformation d'un fichier brut

Prenons un fichier JSON brut et voyons comment il est transformé en lignes du CSV.

In [ ]:
# Charger un fichier brut
sample_raw = sorted(RAW_DIR.glob("*_current_*.json"))[0]
with open(sample_raw) as f:
    raw_data = json.load(f)

print(f"Fichier : {sample_raw.name}")
print(f"Nombre de mesures dans ce fichier : {len(raw_data['list'])} heure(s)")
print(f"Ville : {raw_data['_city_meta']['name']}")
print()

# Appliquer la transformation
rows = _rows_from_file(sample_raw)
df_sample = pd.DataFrame(rows)
print(f"Lignes produites par _rows_from_file : {len(df_sample)}")
df_sample

## 5. Résumé

| Fonction | Rôle |
|---|---|
| `_rows_from_file(path)` | Extrait les mesures d'un fichier JSON brut en dictionnaires |
| `rebuild_clean()` | Parcourt raw/, déduplique, trie, écrit clean/qualite_air.csv |

**Points clés :**
- Reconstruction **totale** à chaque run (pas d'append)
- **Idempotent** : rejouable sans effet de bord
- Déduplication sur (ville, timestamp_utc)
- Tri chronologique par ville
- 8 polluants extraits : CO, NO, NO2, O3, SO2, PM2.5, PM10, NH3
- AQI sur l'échelle OpenWeather 1-5